In [ ]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
# import imageio --- IGNORE IF DEPRECATED---
import imageio.v3 as imageio
import copy
import os
import glob

OpenCV should be 4.8.x Current version: 4.8.0


In [2]:
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

In [3]:

imgs_path = []
for i in range(21):
    path = f"../data/img_{i}.jpg"
 
    imgs_path.append(path)
imgs = load_images(imgs_path)
 

In [5]:
corners = [cv2.findChessboardCorners(img, (10,7)) for img in imgs]

In [6]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

corners_refined = []

for i in range(len(imgs)):
    if corners_copy[i][0]:
        corners_refined.append((True,cv2.cornerSubPix(imgs_gray[i], corners_copy[i][1], (6,6), (-1,-1), criteria)))
    else:
        corners_refined.append((False,None))

In [7]:
imgs_copy = copy.deepcopy(imgs)

In [8]:
for i in range(len(imgs_copy)):
    imgs_copy[i] = cv2.drawChessboardCorners(imgs_copy[i], (10,7), corners_refined[i][1], corners_refined[i][0])

In [9]:
def show_image(img_name, img):
    cv2.imshow(img_name, img)
    cv2.waitKey()
    cv2.destroyAllWindows()
    
def write_image(img_route, img):
    cv2.imwrite(img_route, img)

In [11]:
for img in imgs_copy:
    show_image("Foto",img)

In [12]:
def get_chessboard_points(chessboard_shape, dx, dy):
    points = []

    #The y_axis corresponds to the longer side
    for y in range(chessboard_shape[1]):
        for x in range(chessboard_shape[0]):
            #We move dx by dx along the bord
            #Once a row on the shorter side is completed, the process is repeated with the next row, separated by a distance dy.
            points.append([x*dx, y*dy, 0])

    return np.array(points, dtype=np.float32)

In [13]:
chessboard_points = [get_chessboard_points((10,7), 20, 20) for corner in corners_refined if corner[0]]

In [14]:
# Filter data and get only those with adequate detections
valid_corners = [corner[1] for corner in corners_refined if corner[0]]
# Convert list to numpy array with np.float32 elements
valid_corners = np.array(valid_corners, dtype=np.float32)

In [15]:
image_size = (imgs[0].shape[1], imgs[0].shape[0])
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(chessboard_points, valid_corners,  image_size, None, None)

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

In [16]:
print("Intrinsics:\n", intrinsics)
print("\nDistortion coefficients:\n", dist_coeffs)
print("\nRoot mean squared reprojection error:\n", rms)

Intrinsics:
 [[974.97037331   0.         666.1734429 ]
 [  0.         983.25628149 349.04193281]
 [  0.           0.           1.        ]]

Distortion coefficients:
 [[ 0.13428688 -0.24967541 -0.00522296  0.00946222  0.0694434 ]]

Root mean squared reprojection error:
 0.349438256314162


In [17]:
#AL PROBAR CON EL GITHUB DE CHEMI REVISAR LA CALIDAD DE LA CÁMARA PARA QUE CUADREN CON LA DEL ORDENADOR